 # Assignment 3
 Imports and setting seeds

In [ ]:
import pandas as pd
import sklearn
import numpy
import surprise
import matplotlib
from surprise import Reader
from surprise.model_selection import train_test_split
from surprise import Dataset
from surprise import SVD
from surprise import accuracy
numpy.random.seed(0)
RELEVANCE_THRESHOLD = 3.5

## Data processing and some statistics

In [ ]:
from matplotlib import pyplot as plt

movies_df = pd.read_csv("movies.csv")


num_cols = movies_df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = movies_df.select_dtypes(include=["object"]).columns

rating_df = pd.read_csv("ratings_train.csv")
rating_df['timestamp'] = pd.to_datetime(rating_df['timestamp'], unit='s')

movies_df = movies_df[~movies_df.eq("").any(axis=1)]
rating_df = rating_df[~rating_df.eq("").any(axis=1)]

combined_dataset = rating_df.join(movies_df.set_index('movieId'), on='movieId')

all_categories = set(genre for all_genres in combined_dataset['genres'] for genre in all_genres.split('|'))

for category in all_categories:
    combined_dataset[category] = combined_dataset['genres'].apply(lambda x: 1 if category in x else 0)

combined_dataset['release_year'] = combined_dataset['title'].str.extract(r'\((19[0-9]{2}|20[0-9]{2})\)').astype('Int64')

min_year_per_movie = combined_dataset['timestamp'].dt.year.groupby(combined_dataset['movieId']).transform('min').astype('Int64')

# Fill NaNs in release_year with the smallest year for that movie_id
combined_dataset['release_year'] = combined_dataset['release_year'].fillna(min_year_per_movie)

print("All genres: ", all_categories)
print("Amount of ratings: ", len(combined_dataset['movieId']))
print("Amount of users: ", len(combined_dataset['userId'].unique()))
print("Amount of movies: ", len(combined_dataset['movieId'].unique()))
print("Amount of movies that are rated before release: ", len(combined_dataset[combined_dataset['timestamp'].dt.year < combined_dataset['release_year']]))
print("Interval of dataset: ", combined_dataset["timestamp"].min(), "-", combined_dataset["timestamp"].max())

rating_counts = combined_dataset.groupby('rating')[['rating']].value_counts()
year_counts = combined_dataset.groupby('release_year')[['release_year']].value_counts()
ratings_per_user = combined_dataset.groupby('userId').size()

# Step 2: Count how many users have each rating count
user_rating_distribution = ratings_per_user.value_counts().sort_index()

print("Rating counts: ", rating_counts)
print("Date counts: ", year_counts)
print("Ratings per user counts: ", user_rating_distribution)

# Plot ratings
plt.figure(figsize=(10, 5))
rating_counts.plot(kind='bar')
plt.title('Number of Items by Rating')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('rating_distribution.png')
plt.show()

# Plot release years
plt.figure(figsize=(12, 5))
year_counts.plot(kind='bar')
plt.title('Number of Items by Release Year')
plt.xlabel('Release Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

combined_dataset

## Train - test split

In [ ]:
ratings_test_df = pd.read_csv("ratings_test.csv")

print("Percentage of users in the to fill in dataset that is not present in training: ", len(set(ratings_test_df["userId"]) - set(combined_dataset["userId"]))/len(ratings_test_df["userId"]))

In [ ]:
def split_data_train_test(combined_dataset, size_testset = 100, per_cold_start_users = 0.1, per_test_movies = 0.1):
    # Shuffle dataset
    combined_dataset = combined_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    # Step 1: Select 100 unique users for the test set
    unique_users = combined_dataset['userId'].unique()
    test_user_count = size_testset
    assert test_user_count <= len(unique_users), "Not enough users in dataset to select 100 test users."

    test_users = np.random.choice(unique_users, size=test_user_count, replace=False)

    # Step 2: Select 10% of test users as cold-start (only in test)
    n_cold_users = int(per_cold_start_users * test_user_count)
    cold_users = np.random.choice(test_users, size=n_cold_users, replace=False)
    known_test_users = set(test_users) - set(cold_users)

    # Step 3: Split the data
    cold_user_data = combined_dataset[combined_dataset['userId'].isin(cold_users)]
    non_test_data = combined_dataset[~combined_dataset['userId'].isin(test_users)]
    known_user_data = combined_dataset[combined_dataset['userId'].isin(known_test_users)]

    # Step 4: For known test users, move per_test_movies of their ratings to test set, keep 1 - per_test_movies in training
    def split_user_ratings_time_based(df):
        df = df.sort_values('timestamp')  # Sort by time
        n_test = max(1, int(per_test_movies * len(df)))
        return df.iloc[:-n_test], df.iloc[-n_test:]

    train_parts = []
    test_parts = []

    for user_id, group in known_user_data.groupby('userId'):
        train_chunk, test_chunk = split_user_ratings_time_based(group)
        train_parts.append(train_chunk)
        test_parts.append(test_chunk)

    # Step 5: Combine everything
    df_train = pd.concat([non_test_data] + train_parts).reset_index(drop=True)
    df_test = pd.concat([cold_user_data] + test_parts).reset_index(drop=True)

    test_users_final = set(df_test['userId'])
    train_users_final = set(df_train['userId'])
    cold_user_pct = len(test_users_final - train_users_final) / len(test_users_final)

    print("Total users in test set:", len(test_users_final))
    print("Cold users (only in test):", len(test_users_final - train_users_final))
    print("Percentage of cold users in test set:", round(cold_user_pct * 100, 2), "%")
    return df_train, df_test


In [ ]:
import numpy as np

sorted_on_timestamp = combined_dataset.sort_values('timestamp')

# 3. Split into 80% train and 20% test
split_index = int(0.8 * len(sorted_on_timestamp))
df_train = sorted_on_timestamp.iloc[:split_index]
df_test = sorted_on_timestamp.iloc[split_index:]

print("Percentage of users in test dataset that is not present in training when using (based on time): ", len(set(df_test["userId"]) - set(df_train["userId"]))/len(set(df_test["userId"])))

reader = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(combined_dataset[['userId', 'movieId', 'rating']], reader)
df_train, df_test = train_test_split(data, test_size=0.2)
train_users = set([df_train.to_raw_uid(u) for u in df_train.all_users()])
test_users = set([uid for (uid, _, _) in df_test])
print("Percentage of users in test dataset that is not present in training when using (20-80 split): ", len(test_users - train_users)/len(test_users))


sorted_on_userId = combined_dataset.sort_values('userId')
split_index = int(0.8 * len(sorted_on_userId))
df_train = sorted_on_userId.iloc[:split_index]
df_test = sorted_on_userId.iloc[split_index:]

## Performance testing functions

## Building Recommender Models and Evaluation

### Get score

In [ ]:
def test_model(algo, train_df, test_df):
    # Prepare training data
    reader = Reader(rating_scale=(0.5, 5.0))
    train_data = Dataset.load_from_df(train_df[["userId", "movieId", "rating"]], reader)
    trainset = train_data.build_full_trainset()
    test_data = Dataset.load_from_df(train_df[["userId", "movieId", "rating"]], reader)
    testset = trainset.build_testset()

    # Fit model
    algo.fit(trainset)

    # Predict
    predictions = algo.test(testset)
    rmse = accuracy.rmse(predictions, verbose=False)
    # Evaluate
    return rmse


### Recomendation code

In [ ]:
from collections import defaultdict
import pandas as pd
from surprise import Dataset, Reader, KNNWithMeans

def get_top_watched_movies(df, k=10):
    # Group by movieId and calculate average rating and count
    df_grouped = df.groupby("movieId").agg(
        average_rating=("rating", "mean"),
        count=("rating", "count")
    )
    average_rating = df_grouped[df_grouped["average_rating"] > 3.5]
    # Get top k by count of ratings
    top_k = average_rating.sort_values(by="count", ascending=False).head(k)

    # Return list of (movieId, average_rating) tuples
    return list(zip(top_k.index, top_k["average_rating"]))

def precision_recall_at_k_from_output(rec_df, test_df, k=10, relevance_threshold=3.5):
    # Create mapping: user_id -> set of relevant items in test set
    relevant_items = (
        test_df[test_df["rating"] >= relevance_threshold]
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    precisions = []
    recalls = []

    for _, row in rec_df.iterrows():
        user_id = row["user_id"]
        recommended_items = set(row[[f"recommendation{i+1}" for i in range(k)]].values)
        relevant = relevant_items.get(user_id, set())

        if not relevant:
            continue

        true_positives = recommended_items & relevant
        precision = len(true_positives) / k
        recall = len(true_positives) / len(relevant)

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = sum(precisions) / len(precisions) if precisions else 0.0
    avg_recall = sum(recalls) / len(recalls) if recalls else 0.0

    return avg_precision, avg_recall


def run_model(algo, trainig_df, test_df, full_df, amount_of_pred = 10, fit=True, keep_scores=False):
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(trainig_df[["userId", "movieId", "rating"]], reader)
    trainset = data.build_full_trainset()

    if fit:
        algo.fit(trainset)

    all_movie_ids = set(full_df['movieId'].unique())
    user_rated = trainig_df.groupby('userId')['movieId'].apply(set).to_dict()

    top_n_recs = []
    for user_id in test_df['userId'].unique():
        rated = user_rated.get(user_id, set())
        unrated = all_movie_ids - rated
        assert len(unrated) != 0, "No unrated movies for user {}".format(user_id)
        user_pred = []
        for movie_id in unrated:
            pred = algo.predict(user_id, movie_id)
            if pred.details['was_impossible']:
                continue
            user_pred.append((pred.iid, pred.est))
            user_pred = sorted(user_pred, key=lambda x: x[1], reverse=True)[:amount_of_pred]
        if len(user_pred) < amount_of_pred:
            user_pred = list(set(user_pred + get_top_watched_movies(trainig_df, amount_of_pred)))[:amount_of_pred]
        if keep_scores:
            top_n_recs.append([user_id] + [item for item in user_pred])
        else:
            top_n_recs.append([user_id] + [item[0] for item in user_pred])

    rec_df = pd.DataFrame(top_n_recs, columns=["user_id"] + ["recommendation" + str(item + 1) for item in range(0,amount_of_pred)])

    print("Fast batch recommendations saved to 'user_recommendations.csv'")

    return rec_df

In [ ]:
df_train, df_test = split_data_train_test(combined_dataset, 100, 0.1,0.2)

algo = KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True})
rmse = test_model(algo, df_train, df_test)
print(rmse)

pred = algo.predict(df_test.iloc[0]["userId"], 4)
print(pred)


pred = algo.predict(df_test.iloc[-1]["userId"], 4)
print(pred)

test_model(algo, df_train, df_test)
rec_df = run_model(algo, df_train, df_test, combined_dataset, 10)

k = 10
precision, recall = precision_recall_at_k_from_output(rec_df, df_test, k=k)
print(f"Precision@{k}: {precision:.4f}")
print(f"Recall@{k}: {recall:.4f}")

pred = algo.predict(df_test.iloc[-1]["userId"], 4)
print(pred)

rec_df

In [ ]:
algo = SVD()
rmse = test_model(algo, df_train, df_test)
rec_df = run_model(algo, df_train, df_test, combined_dataset)

k = 10
precision, recall = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=3)
print(f"Precision@{k}: {precision:.4f}")
print(f"Recall@{k}: {recall:.4f}")
print(f"RSME: {rmse:.4f}")
rec_df

### Test configurations KNNMean

In [ ]:
df_train, df_test = split_data_train_test(combined_dataset, 100, 0.1,0.2)
options = [ {'name': 'cosine', 'user_based': True}, {'name': 'cosine', 'user_based': False},
            {'name': 'msd', 'user_based': True}, {'name': 'msd', 'user_based': False},
             {'name': 'pearson', 'user_based': True}, {'name': 'pearson', 'user_based': False},
            {'name': 'pearson_baseline', 'user_based': True}, {'name': 'pearson_baseline', 'user_based': False}]
k_values = [20,50,100]

results = []
for option in options:
    for k_value in k_values:
        algo = KNNWithMeans(k=k_value, sim_options=option)
        rmse = test_model(algo, df_train, df_test)
        rec_df = run_model(algo, df_train, df_test, combined_dataset)

        k = 10
        precision10, recall10 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

        k = 3
        precision3, recall3 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

        k = 1
        precision1, recall1 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

        results.append({
            "similarities": option["name"],
            "method": "KNN",
            "user_based": option["user_based"],
            "k_value": k_value,
            "precision@10": precision10,
            "recall@10": recall3,
            "precision@3": precision3,
            "recall@1": recall3,
            "precision@1": precision1,
            "recall@1": recall1,
            "rmse": rmse,
        })
        print("done")

result_df = pd.DataFrame(results)
result_df = result_df.sort_values("precision@10")
print(result_df.to_latex())
pd.DataFrame(results)


### Test configurations SVD

In [ ]:
learning_rates = [ 0.01, 0.001, 0.0001 ]
epochs = [20,50,100]
n_factors = [50,100,200]

results = []
for n_factor in n_factors:
    for epoch in epochs:
        for lr in learning_rates:
            algo = SVD(n_factors=n_factor, n_epochs=epoch, lr_all=lr, verbose=False)
            rmse = test_model(algo, df_train, df_test)
            rec_df = run_model(algo, df_train, df_test, combined_dataset)

            k = 10
            precision10, recall10 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

            k = 3
            precision3, recall3 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

            k = 1
            precision1, recall1 = precision_recall_at_k_from_output(rec_df, df_test, k=k, relevance_threshold=RELEVANCE_THRESHOLD)

            results.append({
                "method": "SVD",
                "epoch": epoch,
                "lr": lr,
                "n_factor": n_factor,
                "precision@10": precision10,
                "recall@10": recall3,
                "precision@3": precision3,
                "recall@3": recall3,
                "precision@1": precision1,
                "recall@1": recall1,
                "rmse": rmse,
            })

result_df = pd.DataFrame(results)
result_df = result_df.sort_values("precision@10")
print(result_df.to_latex())
pd.DataFrame(results)


## Performance analyse

### Knn

In [ ]:

import tracemalloc
from datetime import datetime


train_time = 0
predict_time = 0
number_of_runs = 4

tracemalloc.clear_traces()
tracemalloc.start()
for _ in range(number_of_runs):
    algo = KNNWithMeans(k=20, sim_options={'name': 'pearson_baseline', 'user_based': True}, verbose=False)
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(df_train[["userId", "movieId", "rating"]], reader)
    trainset = data.build_full_trainset()
    start_time = datetime.now()
    algo.fit(trainset)
    end_time = datetime.now()
    train_time += (end_time - start_time).total_seconds() * 10**3

    start_time = datetime.now()
    rec_df = run_model(algo, df_train, df_test, combined_dataset, fit=False)
    end_time = datetime.now()
    predict_time += (end_time - start_time).total_seconds() * 10**3

memory_usage_knn = tracemalloc.get_traced_memory()
print(f"Average train time: {train_time/number_of_runs} ms, Average predict time: {predict_time/number_of_runs} ms, memory usage: {memory_usage_knn[1]}")
tracemalloc.stop()
train_time_knn = train_time/number_of_runs
predict_time_knn = predict_time/number_of_runs


### SVD

In [ ]:

from datetime import datetime

train_time = 0
predict_time = 0
number_of_runs = 4

tracemalloc.clear_traces()
tracemalloc.start()
for _ in range(number_of_runs):
    algo = SVD(n_factors=50, n_epochs=50, lr_all=0.001, verbose=False)
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(df_train[["userId", "movieId", "rating"]], reader)
    trainset = data.build_full_trainset()
    start_time = datetime.now()
    algo.fit(trainset)
    end_time = datetime.now()
    train_time += (end_time - start_time).total_seconds() * 10**3

    start_time = datetime.now()
    rec_df = run_model(algo, df_train, df_test, combined_dataset, fit=False)
    end_time = datetime.now()
    predict_time += (end_time - start_time).total_seconds() * 10**3

memory_usage_svd = tracemalloc.get_traced_memory()
print(f"Average train time: {train_time/number_of_runs} ms, Average predict time: {predict_time/number_of_runs} ms, memory usage: {memory_usage_svd[1]}")
tracemalloc.stop()

train_time_svd = train_time/number_of_runs
predict_time_svd = predict_time/number_of_runs


In [ ]:
labels = ["SVD", "KNN"]
train_times = [train_time_svd, train_time_knn]
predict_times = [predict_time_svd, predict_time_knn]
memory_usages = [memory_usage_svd[1], memory_usage_knn[1]]

x = range(len(labels))
width = 0.25  # Bar width

fig, ax = plt.subplots()
ax.bar([p - width for p in x], train_times, width, label='Train Time')
ax.bar(x, predict_times, width, label='Predict Time')
ax.set_xlabel('Model')
ax.set_ylabel('Time (ms)')
ax.set_title('Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.tight_layout()
plt.savefig("time_comp.png")
plt.show()

### Average recomended movie score

In [ ]:
from collections import Counter

algo = SVD(n_factors=50, n_epochs=50, lr_all=0.001, verbose=False)
rec_df = run_model(algo, df_train, df_test, combined_dataset, keep_scores=True)

recommendation_columns = [col for col in rec_df.columns if 'recommendation' in col]
all_scores = []

for col in recommendation_columns:
    all_scores.extend(rec_df[col].apply(lambda x: x[1]))

# Step 2: Round scores to nearest 0.5
rounded_scores = [round(score * 2) / 2 for score in all_scores]

# Step 3: Count occurrences
score_counts = Counter(rounded_scores)

# Step 4: Plot
sorted_scores = sorted(score_counts.keys())
frequencies = [score_counts[score] for score in sorted_scores]

plt.figure(figsize=(8, 5))
plt.bar(sorted_scores, frequencies, width=0.4)
plt.title("Score Distribution (Rounded to Nearest 0.5)")
plt.xlabel("Rounded Score")
plt.ylabel("Frequency")
plt.xticks(sorted_scores)  # Ensure all score ticks are shown
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("score_distribution.png")
plt.show()


# Creating ratings_test.csv

In [ ]:
algo = SVD(n_factors=50, n_epochs=50, lr_all=0.001, verbose=False)
rec_df = run_model(algo, combined_dataset, ratings_test_df, combined_dataset)
rec_df["userId"] = rec_df["user_id"]
del rec_df["user_id"]
rec_df.to_csv("ratings.csv", index=False)